# Band Structure with a Custom Pseudopotential

Calculate the electronic band structure of a material with a pseudopotential you provide, driven
entirely from Python. The pseudopotential is uploaded to your object storage folder ("Dropbox"),
a workflow unit fetches it onto the compute node next to the calculation, and the Quantum
ESPRESSO input is pointed at it.

<h2 style="color:green">Usage</h2>

1. Put your pseudopotential file in the `../uploads` folder (a Si example, `Si.upf`, ships with
   this notebook) and set the parameters in cell 1.2. below.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the
   material, the pseudopotential file, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then
   select account and project.
1. Create material: read from the `../uploads` folder, with Standata as a fallback, then saved to
   the platform.
1. Configure workflow: upload the pseudopotential, load the band structure workflow from Standata,
   set the model, and attach a unit that fetches the uploaded file onto the compute node.
1. Configure compute: get list of clusters and create compute configuration.
1. Create the job, then point its Quantum ESPRESSO inputs at the uploaded pseudopotential.
1. Submit the job and monitor the status.
1. Retrieve results: confirm which pseudopotential was used and display the band structure.

## How the custom pseudopotential reaches the calculation

Quantum ESPRESSO reads pseudopotentials from the job's `pseudo` directory (`pseudo_dir` in the
input). Three pieces line up to put your file there and make the calculation use it:

| Piece | What it does |
| --- | --- |
| upload (§4.1) | puts the file in your account's object storage folder |
| io unit (§4.2) | fetches it into the job's `pseudo` directory on the compute node |
| input edit (§6.2) | names the file in the `ATOMIC_SPECIES` card of each pw.x input |

The default pseudopotential the platform would have picked is simply never referenced.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

`PSEUDO_FILE` names a UPF file in the `../uploads` folder — drag your own into the JupyterLite
file browser to replace the shipped example. `PSEUDO_ELEMENT` is the element it is for, and the
model parameters below describe it (the shipped `Si.upf` is norm-conserving PBE).

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 4. Pseudopotential parameters
PSEUDO_FILE = "Si.upf"
PSEUDO_ELEMENT = "Si"

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure - Custom Pseudopotential"

# Model parameters, describing the uploaded pseudopotential
MODEL_SUBTYPE = "gga"        # "gga" or "lda"
FUNCTIONAL = "pbe"           # for gga: "pbe", "pbesol"; for lda: "pz"
PSEUDOPOTENTIAL_TYPE = "nc"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"\u2705 Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"\u2705 Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

material.basis.set_labels_from_list([])
saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Upload the pseudopotential

The file goes to your account's object storage folder, which the compute node fetches it from.
An upload travels as a string inside the request body, which the API caps at 50 MB, and must be
UTF-8 text — UPF files are XML-like text, so they qualify. Anything larger goes to the same
folder through the Dropbox page in the web interface instead.

In [ ]:
import os

from mat3ra.notebooks_utils.core.entity.file.api import upload_files

path = os.path.join(FOLDER, PSEUDO_FILE)
if not os.path.exists(path):
    raise FileNotFoundError(f"'{PSEUDO_FILE}' is not in {FOLDER}. Drag it into the file browser first.")
with open(path, "rb") as file:
    content = file.read().decode("utf-8")

uploaded_pseudo = upload_files(client, {PSEUDO_FILE: content}, ACCOUNT_ID)[0]

### 4.2. Load workflow from Standata, attach the fetch of the pseudopotential, and preview

An `io` unit is placed at the head of the workflow before it is built. It downloads the uploaded
file into the job's `pseudo` subdirectory — the exact directory `pseudo_dir` in the Quantum
ESPRESSO inputs points at — before the first pw.x unit runs.

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.core.entity.file.api import to_object_storage_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
subworkflow = workflow_config["subworkflows"][0]
first_unit = subworkflow["units"][0]
io_unit = {
    "type": "io",
    "name": "io-pseudo",
    "subtype": "input",
    "source": "object_storage",
    "input": [to_object_storage_input(uploaded_pseudo, pathname="pseudo")],
    # Nothing references this id, but a unit needs one to sit in the flowchart; namespaced by
    # workflow so it cannot clash.
    "flowchartId": "band-structure-custom-pseudo-io",
    "head": True,
    "next": first_unit["flowchartId"],
    "status": "idle",
    "statusTrack": [],
    "results": [],
    "monitors": [],
    "preProcessors": [],
    "postProcessors": [],
}
first_unit["head"] = False
subworkflow["units"].insert(0, io_unit)

workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Set the model to match the uploaded pseudopotential

The method subtype tells the platform what kind of pseudopotential the calculation uses, and the
cutoffs should suit it — norm-conserving potentials typically want a higher wavefunction cutoff
than ultrasoft ones.

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode import ModelFactory
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True
).get_context_item_data()
for unit_name in ["pw_scf", "pw_bands"]:
    unit = workflow.subworkflows[0].get_unit_by_name(name=unit_name)
    if unit:
        unit.add_context(cutoffs_context)
        workflow.subworkflows[0].set_unit(unit)

### 4.4. Save workflow to collection

In [ ]:
saved_workflow = client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
workflow_id = saved_workflow["_id"]
print(f"✅ Workflow saved to collection: {workflow_id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

job_name = MY_WORKFLOW_NAME + " " + saved_material.formula + " " + timestamp
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=saved_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job_id = job_response["_id"]
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

### 6.2. Point the inputs at the uploaded pseudopotential

At job creation the platform fills the `ATOMIC_SPECIES` card of each pw.x input with its default
pseudopotential for the element. The lines are edited here to name the uploaded file instead —
the expert-mode input edit the platform documentation describes — and `isManuallyChanged` keeps
the edit from being re-rendered.

In [ ]:
import re

job = client.jobs.get(job_id)
# Element + mass + filename, matched only inside the ATOMIC_SPECIES card - the same
# "element number number" shape also occurs in ATOMIC_POSITIONS, where a bare substitution
# would overwrite a coordinate.
pattern = re.compile(rf"^(\s*{PSEUDO_ELEMENT}\s+[\d.]+\s+)\S+", re.MULTILINE)

for unit in job["workflow"]["subworkflows"][0]["units"]:
    for unit_input in unit.get("input", []):
        rendered = unit_input.get("rendered") or ""
        if "ATOMIC_SPECIES" not in rendered:
            continue
        start = rendered.index("ATOMIC_SPECIES")
        end = rendered.find("\n\n", start)
        end = len(rendered) if end == -1 else end
        card = pattern.sub(rf"\g<1>{PSEUDO_FILE}", rendered[start:end])
        unit_input["rendered"] = rendered[:start] + card + rendered[end:]
        unit_input["isManuallyChanged"] = True
        print(f"✅ {unit['name']} now uses {PSEUDO_FILE}")

client.jobs.update(job_id, job)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Confirm the pseudopotential that was used

The `ATOMIC_SPECIES` card of each pw.x input, exactly as the calculation consumed it.

In [ ]:
finished_job = client.jobs.get(job_id)
for unit in finished_job["workflow"]["subworkflows"][0]["units"]:
    for unit_input in unit.get("input", []):
        rendered = unit_input.get("rendered") or ""
        if "ATOMIC_SPECIES" in rendered:
            card = rendered[rendered.index("ATOMIC_SPECIES"):].split("\n\n")[0]
            print(f"--- {unit['name']} ---\n{card}")

### 8.2. Band Structure

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structure_data = get_properties_for_job(client, job_id, property_name=PropertyName.non_scalar.band_structure.value)
visualize_properties(band_structure_data, title="Band Structure", extra_config={"material": material.to_dict()})